# 02. Feature Importance, Market Regimes, and Project Limitations
Welcome to the second exploratory notebook of the **TSX Stock Price Direction Predictor**.

### Objectives:
1. **Feature Importance Analysis**: Extract and compare feature importances across Random Forest and XGBoost to identify which market indicators actually drive predictions.
2. **Regime Analysis**: Evaluate how model accuracy, baseline performance, and volatility vary across distinct historical regimes:
   - **Pre-COVID Tranquil Market** (2017 – 2019)
   - **2020 COVID Shock & Stimulus Rebound** (Feb 2020 – Dec 2020)
   - **2022 Central Bank Rate Hike Cycle** (Jan 2022 – Dec 2022)
   - **2024–2026 Out-of-Sample Holdout Period** (Oct 2024 – Oct 2026)
3. **Honest Limitations Section**: A critical evaluation covering:
   - **Overfitting Risk** in low signal-to-noise environments
   - **Market Efficiency** and why beating naive baselines is challenging
   - **Survivorship Bias** in stock universe selection

In [ ]:
import sys
from pathlib import Path
import joblib
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import accuracy_score

project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.data import ALL_DEFAULT_TICKERS, DEFAULT_STOCKS, BENCHMARK_INDEX
from src.features import load_processed_features

sns.set_theme(style="whitegrid", palette="tab10")
plt.rcParams["figure.figsize"] = (12, 6)
plt.rcParams["font.size"] = 10

print("Libraries imported successfully.")
print(f"Project root: {project_root}")

### 1. Feature Importance in Tree Ensembles (Random Forest & XGBoost)
Tree models compute feature importance by measuring the total reduction in impurity (Gini / Entropy for Random Forest, and Gain for XGBoost) brought by each feature across all split nodes.

Let's inspect:
- Which features provide the most predictive power?
- Do tree models favor technical indicators (RSI, Moving Averages), momentum, volatility, or cross-market index context?

In [ ]:
# Extract feature importances across all assets and tree models
rf_importances = {}
xgb_importances = {}

models_dir = project_root / "data" / "models"

for ticker in ALL_DEFAULT_TICKERS:
    clean_t = ticker.replace("^", "")
    df = load_processed_features(ticker, processed_dir=project_root / "data" / "processed")
    feat_cols = [c for c in df.columns if c not in ["target", "target_return"]]
    
    rf_path = models_dir / f"{clean_t}_random_forest.joblib"
    xgb_path = models_dir / f"{clean_t}_xgboost.joblib"
    
    if rf_path.exists():
        rf = joblib.load(rf_path)
        rf_importances[ticker] = pd.Series(rf.feature_importances_, index=feat_cols)
        
    if xgb_path.exists():
        xgb = joblib.load(xgb_path)
        xgb_importances[ticker] = pd.Series(xgb.feature_importances_, index=feat_cols)

rf_imp_df = pd.DataFrame(rf_importances)
xgb_imp_df = pd.DataFrame(xgb_importances)

# Compute mean importance across equities
rf_mean = rf_imp_df.mean(axis=1).sort_values(ascending=False)
xgb_mean = xgb_imp_df.mean(axis=1).sort_values(ascending=False)

print("Top 5 Random Forest Features (Mean across all assets):")
print(rf_mean.head(5).round(4))
print()
print("Top 5 XGBoost Features (Mean across all assets):")
print(xgb_mean.head(5).round(4))

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 7), sharey=False)

# Random Forest Plot
y_pos = np.arange(len(rf_mean))
ax1.barh(y_pos, rf_mean.values, color="forestgreen", alpha=0.85)
ax1.set_yticks(y_pos)
ax1.set_yticklabels(rf_mean.index, fontsize=10)
ax1.invert_yaxis()
ax1.set_xlabel("Mean Impurity Reduction (Gini)")
ax1.set_title("Random Forest: Feature Importance Ranking", fontsize=12, fontweight="bold")
ax1.grid(True, linestyle="--", alpha=0.5)

# XGBoost Plot
y_pos2 = np.arange(len(xgb_mean))
ax2.barh(y_pos2, xgb_mean.values, color="crimson", alpha=0.85)
ax2.set_yticks(y_pos2)
ax2.set_yticklabels(xgb_mean.index, fontsize=10)
ax2.invert_yaxis()
ax2.set_xlabel("Mean Gain")
ax2.set_title("XGBoost: Feature Importance Ranking", fontsize=12, fontweight="bold")
ax2.grid(True, linestyle="--", alpha=0.5)

plt.suptitle("Cross-Asset Feature Importance Comparison", fontsize=14, fontweight="bold", y=0.99)
plt.tight_layout()
plt.show()

### 2. Performance Across Distinct Macroeconomic Regimes
Financial models rarely perform uniformly across all market cycles. A model that performs well during quiet bull runs can suffer deep drawdowns when market volatility surges or central banks shift rates.

We segment our 10-year dataset into 4 distinct historical regimes:
1. **Pre-COVID Tranquil** (2017-01-01 to 2019-12-31): Steady economic expansion, low volatility (~10–14% annualized).
2. **2020 COVID Shock & Rebound** (2020-02-01 to 2020-12-31): Extreme volatility spike (>35–50%), rapid market crash followed by unprecedented monetary stimulus.
3. **2022 Central Bank Rate Hike Cycle** (2022-01-01 to 2022-12-31): Persistent inflation, aggressive rate increases, tech valuation compression.
4. **2024–2026 Holdout Test Period** (2024-10-29 to 2026-10-15): Unseen test period, commodity rotation, strong Canadian bank performance.

In [ ]:
regimes = {
    "2017-2019 Tranquil": ("2017-01-01", "2019-12-31"),
    "2020 COVID Shock": ("2020-02-01", "2020-12-31"),
    "2022 Rate Hikes": ("2022-01-01", "2022-12-31"),
    "2024-2026 Test Period": ("2024-10-29", "2026-10-15"),
}

regime_records = []

for ticker in ALL_DEFAULT_TICKERS:
    clean_t = ticker.replace("^", "")
    df = load_processed_features(ticker, processed_dir=project_root / "data" / "processed")
    feat_cols = [c for c in df.columns if c not in ["target", "target_return"]]
    
    lr = joblib.load(models_dir / f"{clean_t}_logistic_regression.joblib")
    rf = joblib.load(models_dir / f"{clean_t}_random_forest.joblib")
    xgb = joblib.load(models_dir / f"{clean_t}_xgboost.joblib")
    
    for r_name, (start, end) in regimes.items():
        mask = (df.index >= start) & (df.index <= end)
        sub_X = df.loc[mask, feat_cols]
        sub_y = df.loc[mask, "target"]
        sub_ret = df.loc[mask, "target_return"]
        
        if len(sub_X) == 0:
            continue
            
        up_rate = (sub_y == 1).mean()
        ann_vol = sub_ret.std() * np.sqrt(252)
        lr_acc = accuracy_score(sub_y, lr.predict(sub_X))
        rf_acc = accuracy_score(sub_y, rf.predict(sub_X))
        xgb_acc = accuracy_score(sub_y, xgb.predict(sub_X))
        
        regime_records.append({
            "Ticker": ticker,
            "Regime": r_name,
            "Trading Days": len(sub_X),
            "Annual Volatility": ann_vol,
            "Always Up Hurdle": up_rate,
            "LR Accuracy": lr_acc,
            "RF Accuracy": rf_acc,
            "XGB Accuracy": xgb_acc,
        })

regime_df = pd.DataFrame(regime_records)

# Compute macro average by regime across all 6 assets
regime_macro = regime_df.groupby("Regime")[
    ["Annual Volatility", "Always Up Hurdle", "LR Accuracy", "RF Accuracy", "XGB Accuracy"]
].mean()

regime_macro_disp = regime_macro.copy()
for col in regime_macro_disp.columns:
    regime_macro_disp[col] = (regime_macro_disp[col] * 100).map("{:.2f}%".format)

print("Macro Performance Summary by Regime (Averaged across all assets):")
regime_macro_disp

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))

# Panel 1: Volatility vs. Baseline Up Rate
x_labels = regime_macro.index
vol_vals = regime_macro["Annual Volatility"] * 100
up_vals = regime_macro["Always Up Hurdle"] * 100

x = np.arange(len(x_labels))
width = 0.35

ax1.bar(x - width/2, vol_vals, width, label="Annualized Volatility (%)", color="dimgray", alpha=0.8)
ax1.bar(x + width/2, up_vals, width, label="Always Up Hurdle (%)", color="royalblue", alpha=0.8)
ax1.set_xticks(x)
ax1.set_xticklabels(x_labels, rotation=15, ha="right")
ax1.set_ylabel("Percentage (%)")
ax1.set_title("Market Dynamics Across Regimes: Volatility & Up-Day Drift", fontsize=11, fontweight="bold")
ax1.legend(loc="upper right")
ax1.grid(True, linestyle="--", alpha=0.5)

# Panel 2: Model Accuracy vs. Always Up Hurdle
rf_acc = regime_macro["RF Accuracy"] * 100
xgb_acc = regime_macro["XGB Accuracy"] * 100

ax2.plot(x_labels, up_vals, "k--", marker="o", linewidth=2.0, label="Always Up Hurdle")
ax2.plot(x_labels, rf_acc, color="forestgreen", marker="s", linewidth=2.0, label="Random Forest Acc")
ax2.plot(x_labels, xgb_acc, color="crimson", marker="^", linewidth=2.0, label="XGBoost Acc")
ax2.set_ylabel("Accuracy (%)")
ax2.set_title("Model Accuracy vs. Naive Hurdle by Market Cycle", fontsize=11, fontweight="bold")
ax2.set_xticks(range(len(x_labels)))
ax2.set_xticklabels(x_labels, rotation=15, ha="right")
ax2.legend(loc="upper right")
ax2.grid(True, linestyle="--", alpha=0.5)

plt.suptitle("Impact of Macro Regimes on Stock Direction Predictability", fontsize=14, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

### 3. Critical Limitations & Reality Check

In quantitative finance and data science, an honest and rigorous analysis of **why models fail** is far more valuable than an inflated accuracy score. When discussing this project in an interview or portfolio review, highlighting these three core limitations demonstrates true domain maturity:

---

#### Limitation 1: Overfitting Risk & Low Signal-to-Noise Ratio
* **The Noise Ceiling**: Daily equity returns are overwhelmingly dominated by market noise, order flow microstructure, unexpected news headlines, and macro announcements. In financial economics, the predictable signal in daily returns rarely exceeds **$R^2 < 1-2\%$**.
* **In-Sample Memorization vs. Out-of-Sample Decay**: Notice in our regime chart how tree models attained **~62–64% accuracy** on training data (2020 and 2022 regimes), but on the completely untouched **2024–2026 test set**, accuracy compressed back to **50–55%**. 
* **Defense Implemented**:
  * We restricted tree depths to shallow limits (`max_depth`: 2–5).
  * We used `TimeSeriesSplit` walk-forward cross-validation without shuffling.
  * We maintained a strictly untouched 20% holdout test period.

---

#### Limitation 2: Market Efficiency (EMH) & Economic Friction
* **Fama's Weak-Form Efficiency**: The Efficient Market Hypothesis states that public historical price and volume data are instantly digested by institutional algorithms, hedge funds, and high-frequency market makers. Simple technical indicators (RSI, moving average crosses) cannot consistently produce abnormal risk-adjusted returns (alpha) in public equities without structural alpha.
* **The Drag of Transaction Friction**: 
  * In our backtest, models that flipped positions 80–130 times incurred **$8.0\%$ to $13.3\%$ in frictional cost drag** at just 0.1% per trade.
  * Even when XGBoost achieved a nominal win rate of 54%, frequent trading friction eroded the strategy's equity below a passive Buy-and-Hold benchmark.
  * Any real-world implementation must include realistic commission, bid-ask spreads, and slippage assumptions.

---

#### Limitation 3: Survivorship Bias
* **The Survivor Universe**: Our asset universe (`RY.TO`, `TD.TO`, `SHOP.TO`, `ENB.TO`, `CNQ.TO`) was hand-picked in **2026** based on their current prominent standing as multi-billion dollar mega-cap leaders on the Toronto Stock Exchange.
* **The Hidden Casualties**: By selecting winners in 2026 and evaluating them backward to 2016, we systematically ignore:
  * Canadian companies that went bankrupt, were forced into distressed restructuring, or suffered catastrophic declines over the past decade.
  * Speculative junior mining and energy companies that delisted from the TSX.
* **The Impact on Baselines**: Survivorship bias artificially inflates historical Buy-and-Hold returns and the "Always Up" baseline (56–59% Up days). If evaluated across the entire historical TSX composite including delisted entities, the naive long drift would be lower, and portfolio turnover costs would be even more punishing.